In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data reading**

In [0]:
df = spark.read.format("parquet")\
            .load("abfss://bronzestg@storageloweretep1.dfs.core.windows.net/products")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.createOrReplaceTempView("products")

### **Functions**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databrickscatalogetep1.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE 
LANGUAGE SQL
RETURN p_price*0.90

In [0]:
%sql
select product_id,price, databrickscatalogetep1.bronze.discount_func(price) as discounted_price from products

In [0]:
df = df.withColumn("discountedPrice",expr("databrickscatalogetep1.bronze.discount_func(price)"))

In [0]:
%sql
 CREATE OR REPLACE FUNCTION databrickscatalogetep1.bronze.upper_func(p_brand STRING)
 RETURNS STRING
 LANGUAGE PYTHON
 AS 
 $$
    return p_brand.upper( )
 $$

In [0]:
%sql
select product_id,brand, databrickscatalogetep1.bronze.upper_func(brand) as discounted_price from products

In [0]:
df.write.format("delta")\
        .mode("append")\
        .option("path","abfss://silverstg@storageloweretep1.dfs.core.windows.net/products")\
        .save()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databrickscatalogetep1.silver.productSilver
        USING DELTA 
        LOCATION 'abfss://silverstg@storageloweretep1.dfs.core.windows.net/products'